In [4]:
import shutil
from pathlib import Path

import numpy as np
import pyceres
import pycolmap
from pycolmap import cost_functions, logging

import enrich_data


def banner(text):
    print(f"\n{'=' * 76}\n{text}\n{'=' * 76}")


def true_cam_from_world(label: str) -> pycolmap.Rigid3d:
    """Ground-truth pose from cameras.csv, converted to COLMAP axes."""
    return enrich_data.cam_from_world(truth, label)


FORCE_OVERWRITE_DB = False  # set to True to overwrite the database if it exists

ROOT = Path.cwd()
database_path = ROOT / "data/ENRICH-Aerial/nadir-2/colmap/database.db"
image_path = ROOT / "data/ENRICH-Aerial/nadir-2/images"
gt_path = ROOT
sfm_free_path = ROOT / "data/ENRICH-Aerial/nadir-2/colmap/sparse"
sfm_gcp_path = ROOT / "data/ENRICH-Aerial/nadir-2/colmap/sparse_gcp"


logging.set_log_destination(logging.INFO, ROOT / "INFO.log")


def remove_existing_database(db_path: Path):
    """Remove the existing database and its associated files."""
    if db_path.exists():
        db_path.unlink()
        # Remove also the wal and shm files if they exist
        wal_path = db_path.with_suffix(".db-wal")
        if wal_path.exists():
            wal_path.unlink()
        shm_path = db_path.with_suffix(".db-shm")
        if shm_path.exists():
            shm_path.unlink()


In [5]:
# `load()` reads the CSVs as they are; `add_noise()` returns another object of
# the same type with a simulated survey applied. `truth` is kept for scoring.

truth = enrich_data.load(
    subblock="nadir-2", split_control_check=True
)  # ground truth, no noise
data = enrich_data.load(
    subblock="nadir-2", split_control_check=True
)  # Use the original data without noise for first run
# data = enrich_data.add_noise(truth)
print(enrich_data.summary(data))

# --- names used further down ------------------------------------------------
cameras_gt = truth.cameras
true_center = truth.camera_xyz
truth_xyz = truth.marker_xyz

surveyed_xyz = data.marker_xyz
marker_data = data.marker_data  # columns: image_name, gcp_name, u, v
camera_centers = data.camera_xyz  # bootstraps the Sim3 alignment only

marker_names = data.marker_names
control_names = data.control_names
check_names = data.check_names

gcp_sigma_m = enrich_data.gcp_sigma_m
gcp_sigma_px = enrich_data.gcp_sigma_px
camera_position_sigma_m = enrich_data.camera_position_sigma_m
tie_points_sigma_px = enrich_data.tie_points_sigma_px

print(
    f"survey noise: xy {gcp_sigma_m[0] * 100:.0f} cm, "
    f"z {gcp_sigma_m[2] * 100:.0f} cm; marks {gcp_sigma_px} px"
)
print(f"GCPs: {control_names}")
print(f"CP   {check_names}")

156 marker image marks, 26 markers, 39 images
control: 18 points | check: 8 points (gcp.001, gcp.004, gcp.010, gcp.013, gcp.016, gcp.019, gcp.022, gcp.025)
survey noise: xy 2 cm, z 3 cm; marks 1.0 px
GCPs: ['gcp.002', 'gcp.003', 'gcp.005', 'gcp.006', 'gcp.007', 'gcp.008', 'gcp.009', 'gcp.011', 'gcp.012', 'gcp.014', 'gcp.015', 'gcp.017', 'gcp.018', 'gcp.020', 'gcp.021', 'gcp.023', 'gcp.024', 'gcp.026']
CP   ['gcp.001', 'gcp.004', 'gcp.010', 'gcp.013', 'gcp.016', 'gcp.019', 'gcp.022', 'gcp.025']


In [6]:
banner("Running COLMAP feature extraction, matching")

if FORCE_OVERWRITE_DB:
    remove_existing_database(database_path)

# SIMPLE_PINHOLE has a SINGLE focal parameter. With PINHOLE, fx and fy are
# refined independently and on this flat nadir block they diverged to
# fx=3227.6 / fy=3818.1 - an 18% anisotropic stretch. That is not a similarity,
# so no alignment can undo it and the whole block ends up distorted by ~8.5 m.
# The renders have no lens distortion (README), so a distortion-free model with
# one focal is the correct choice - this is also what Metashape used.
readerOptions = pycolmap.ImageReaderOptions(camera_model="SIMPLE_PINHOLE")
extraction_options = pycolmap.FeatureExtractionOptions(
    type=pycolmap.FeatureExtractorType.SIFT,
    max_image_size=-1,
    use_gpu=False,
    sift=pycolmap.SiftExtractionOptions(
        max_num_features=8192,  # 4096,  # default is 8192
    ),
)
matching_options = pycolmap.FeatureMatchingOptions(
    type=pycolmap.FeatureMatcherType.SIFT_BRUTEFORCE,
    use_gpu=False,
)

pycolmap.extract_features(
    database_path,
    image_path,
    camera_mode=pycolmap.CameraMode.SINGLE,
    reader_options=readerOptions,
    extraction_options=extraction_options,
)

pycolmap.match_exhaustive(
    database_path,
    matching_options=matching_options,
)


Running COLMAP feature extraction, matching


W20260812 16:26:04.760476 558585 feature_extraction.cc:445] Your current options use the maximum number of threads on the machine to extract features. Extracting SIFT features on the CPU can consume a lot of RAM per thread for large images. Consider reducing the maximum image size and/or the first octave or manually limit the number of extraction threads. Ignore this warning, if your machine has sufficient memory for the current settings.
I20260812 16:26:04.760651 559356 feature_extraction.cc:494] === Feature extraction ===
I20260812 16:26:04.761178 559373 sift.cc:765] Creating SIFT CPU feature extractor
I20260812 16:26:04.761221 559374 sift.cc:765] Creating SIFT CPU feature extractor
I20260812 16:26:04.761243 559376 sift.cc:765] Creating SIFT CPU feature extractor
I20260812 16:26:04.761225 559375 sift.cc:765] Creating SIFT CPU feature extractor
I20260812 16:26:04.761267 559377 sift.cc:765] Creating SIFT CPU feature extractor
I20260812 16:26:04.761300 559378 sift.cc:765] Creating SIFT 

In [7]:
banner("Seed the camera with the known intrinsics")

db = pycolmap.Database.open(database_path)
try:
    for camera in db.read_all_cameras():
        camera.model = pycolmap.CameraModelId.SIMPLE_PINHOLE
        camera.width, camera.height = enrich_data.IMAGE_WIDTH, enrich_data.IMAGE_HEIGHT
        camera.params = np.array([enrich_data.FOCAL_PX, *enrich_data.PRINCIPAL_POINT])
        camera.has_prior_focal_length = True
        db.update_camera(camera)
        print(f"camera {camera.camera_id}: {camera.model.name} {camera.params}")
finally:
    db.close()


Seed the camera with the known intrinsics
camera 1: SIMPLE_PINHOLE [5865.1810585 3008.        2008.       ]


In [16]:
# banner("Seed the camera with the known camera positions")

# db = pycolmap.Database.open(database_path)
# try:
#     for image in db.read_all_images():
#         prior = pycolmap.PosePrior()
#         prior.corr_data_id = image.data_id
#         prior.position = camera_centers[image.name]
#         prior.position_covariance = np.diag([camera_position_sigma_m**2] * 3)
#         prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
#         db.write_pose_prior(prior)

#     if db.num_pose_priors() < 3:
#         print(
#             f"WARNING: only {db.num_pose_priors()} pose priors; COLMAP needs >= 3 "
#             "or it silently ignores them and fixes the gauge instead."
#         )
# finally:
#     db.close()


In [8]:
banner("Running COLMAP incremental mapping")

options = pycolmap.IncrementalPipelineOptions()
options.use_prior_position = True
# A flat nadir block cannot determine focal length from imagery:
# Hold the intrinsics at the known values; the GCP bundle can free them later, where the control breaks the degeneracy.
options.ba_refine_focal_length = False
options.ba_refine_principal_point = False
options.ba_refine_extra_params = False

# --- tie-point filtering during mapping -------------------------------------
# Stated explicitly (these are COLMAP's own defaults) so the tuning surface is
# visible rather than implicit. All three are NO-OPS on this block - the 80/60
# overlap gives strong geometry - but they matter on weaker blocks.
# Measured here: triangulation angle p1 = 7.4 deg, median 26.8 deg, ZERO points
# below 5 deg; reprojection error max 3.98 px.
options.mapper.filter_max_reproj_error = 4.0  # no-op here: max observed 3.98 px
options.mapper.filter_min_tri_angle = 1.5  # degrees; no-op, p1 is 7.4 deg
options.mapper.ba_local_min_tri_angle = 6.0  # degrees; no-op, p1 is 7.4 deg

# Start clean so the previously distorted reconstruction is never reused.
if sfm_free_path.exists():
    shutil.rmtree(sfm_free_path)
sfm_free_path.mkdir(parents=True, exist_ok=True)

recs = pycolmap.incremental_mapping(
    database_path, image_path, sfm_free_path, options=options
)
rec = recs[max(recs, key=lambda k: recs[k].num_reg_images())]
print(rec.summary())
print("camera:", list(rec.cameras.values())[0].params)


rec.write(sfm_free_path)  # write the best reconstruction to disk


Running COLMAP incremental mapping


I20260812 16:26:11.309048 558585 incremental_pipeline.cc:348] Loading database
I20260812 16:26:11.309118 558585 database_cache.cc:73] Loading rigs...
I20260812 16:26:11.309145 558585 database_cache.cc:83]  1 in 0.000s
I20260812 16:26:11.309156 558585 database_cache.cc:91] Loading cameras...
I20260812 16:26:11.309171 558585 database_cache.cc:109]  1 in 0.000s
I20260812 16:26:11.309180 558585 database_cache.cc:117] Loading frames...
I20260812 16:26:11.309243 558585 database_cache.cc:127]  39 in 0.000s
I20260812 16:26:11.309253 558585 database_cache.cc:135] Loading matches...
I20260812 16:26:11.314810 558585 database_cache.cc:140]  441 in 0.006s
I20260812 16:26:11.314852 558585 database_cache.cc:148] Loading images...
I20260812 16:26:11.329154 558585 database_cache.cc:245]  39 in 0.014s (connected 39, loaded 39)
I20260812 16:26:11.329202 558585 database_cache.cc:259] Loading pose priors...
I20260812 16:26:11.329236 558585 database_cache.cc:267]  39 in 0.000s
I20260812 16:26:11.329242 5585

Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 39
	num_reg_frames = 39
	num_images = 39
	num_points3D = 43754
	num_observations = 221691
	mean_track_length = 5.06676
	mean_observations_per_image = 5684.38
	mean_reprojection_error = 0.607466
camera: [5865.1810585 3008.        2008.       ]


I20260812 16:26:37.505254 558585 incremental_pipeline.cc:793] Keeping successful reconstruction
I20260812 16:26:37.550140 558585 timer.cc:90] Elapsed time: 0.436 [minutes]


In [9]:
banner("Coarse Sim3 alignment (starting point only)")


def camera_center_rmse(reconstruction):
    err = [
        np.linalg.norm(im.projection_center() - true_center[im.name])
        for im in reconstruction.images.values()
    ]
    return float(np.sqrt(np.mean(np.square(err))))


def umeyama(P, Q):
    """Least-squares similarity mapping P -> Q over ALL correspondences.

    align_reconstruction_to_locations uses RANSAC on 3-point samples, which is
    badly conditioned here: the camera centres are essentially coplanar (true
    altitude is a constant 188 m), so it returned transforms tilted ~21 deg off
    and 40-170 m out. A plain least-squares fit over all 39 is deterministic
    and well behaved.
    """
    mp, mq = P.mean(0), Q.mean(0)
    X, Y = P - mp, Q - mq
    U, S, Vt = np.linalg.svd(X.T @ Y / len(P))
    D = np.eye(3)
    D[2, 2] = np.sign(np.linalg.det(U @ Vt))
    R = (U @ D @ Vt).T
    s = (S * np.diag(D)).sum() / (X**2).sum() * len(P)
    return pycolmap.Sim3d(s, pycolmap.Rotation3d(R), mq - s * R @ mp)


# names = [im.name for im in rec.images.values()]
# sim3 = umeyama(
#     np.array([im.projection_center() for im in rec.images.values()]),
#     np.array([camera_centers[n] for n in names]),
# )
# rec.transform(sim3)
# print(f"Sim3, scale = {sim3.scale:.6f}")

rmse = camera_center_rmse(rec)
print(f"camera centre RMSE after alignment: {rmse:.4f} m")

# A similarity absorbs uniform scale but NOT anisotropic distortion. A large
# residual here means the free network is deformed - almost always a runaway
# self-calibration - and no amount of GCP control will rescue it.
# assert rmse < 1, (
#     f"free network is distorted ({rmse:.2f} m residual after a best-fit "
#     "similarity). Check that the camera has a single focal and that "
#     "ba_refine_focal_length was disabled during mapping."
# )


Coarse Sim3 alignment (starting point only)
camera centre RMSE after alignment: 0.0153 m


In [11]:
def _stats(values):
    """mean / median / RMS / p90 / p99 / max of a 1-D array."""
    v = np.asarray(values, dtype=float)
    return {
        "mean": v.mean(),
        "median": np.median(v),
        "rms": np.sqrt((v**2).mean()),
        "p90": np.percentile(v, 90),
        "p99": np.percentile(v, 99),
        "max": v.max(),
    }


def block_report(reconstruction, marker_obs=None, marker_xyz=None):
    """Internal quality of the block.

    These are the numbers available on a real project: they say whether the
    block is well conditioned and self-consistent, but nothing about absolute
    accuracy - for that see accuracy_report().
    """
    banner("BLOCK QUALITY")

    camera = next(iter(reconstruction.cameras.values()))
    n_img = reconstruction.num_reg_images()
    print(
        f"  images            {n_img} registered / {len(reconstruction.images)} in "
        f"block | {len(reconstruction.cameras)} camera(s) {camera.model.name}"
    )
    print(f"  intrinsics        {np.round(camera.params, 3)}")

    points = list(reconstruction.points3D.values())
    track = np.array([p.track.length() for p in points])
    error = np.array([p.error for p in points])
    n_obs = int(track.sum())
    print(f"  points3D          {len(points):,}      observations {n_obs:,}")

    # The histogram matters more than the mean: 2-ray points are the weak ones.
    bins = [(2, 2), (3, 5), (6, 9), (10, 10**6)]
    hist = "   ".join(
        f"{lo}{'' if lo == hi else ('+' if hi > 1000 else f'-{hi}')}: "
        f"{((track >= lo) & (track <= hi)).sum():,} "
        f"({100 * ((track >= lo) & (track <= hi)).mean():.1f}%)"
        for lo, hi in bins
    )
    print(
        f"  track length      mean {track.mean():.2f}   median {int(np.median(track))}"
    )
    print(f"                    {hist}")

    per_image = np.array([im.num_points3D for im in reconstruction.images.values()])
    print(
        f"  observations/img  min {per_image.min():,}   "
        f"mean {per_image.mean():,.0f}   max {per_image.max():,}"
    )

    s = _stats(error)
    print("\n  tie-point reprojection error (px)")
    print(
        f"    mean {s['mean']:.3f}   median {s['median']:.3f}   RMS {s['rms']:.3f}   "
        f"p90 {s['p90']:.3f}   p99 {s['p99']:.3f}   max {s['max']:.2f}"
    )
    print(f"    > 2 px: {(error > 2).sum():,} points ({100 * (error > 2).mean():.1f}%)")

    # Standard photogrammetric redundancy: each observation gives 2 equations,
    # each image 6 pose unknowns, each point 3, plus the intrinsics.
    n_unknowns = 6 * n_img + 3 * len(points) + len(camera.params)
    print(
        f"\n  redundancy        {2 * n_obs:,} measurements - {n_unknowns:,} "
        f"unknowns  ->  r = {2 * n_obs - n_unknowns:,}"
    )

    # Marker observations are weighted differently from tie points and are how
    # the control acts on the block, so score them separately.
    if marker_obs is None or marker_xyz is None:
        return
    by_name = {im.name: im for im in reconstruction.images.values()}
    residuals, per_marker = [], {}
    for obs in marker_obs.itertuples():
        image = by_name.get(obs.image_name)
        if image is None or obs.marker_name not in marker_xyz:
            continue
        cam = reconstruction.cameras[image.camera_id]
        uv = cam.img_from_cam(image.cam_from_world() * marker_xyz[obs.marker_name])
        if uv is None:
            continue
        d = float(np.hypot(uv[0] - obs.u, uv[1] - obs.v))
        residuals.append(d)
        per_marker.setdefault(obs.marker_name, []).append(d)

    if residuals:
        s = _stats(residuals)
        print(
            f"\n  marker observations ({len(residuals)} in "
            f"{marker_obs.image_name.nunique()} images)"
        )
        print(
            f"    reprojection  mean {s['mean']:.2f} px   "
            f"median {s['median']:.2f}   max {s['max']:.2f}"
        )
        worst = sorted(per_marker.items(), key=lambda kv: -np.mean(kv[1]))[:5]
        print("    worst:  " + ",  ".join(f"{n} {np.mean(v):.2f} px" for n, v in worst))


def accuracy_report(reconstruction, marker_xyz, truth, label=""):
    """External accuracy against ground truth.

    Reports per-axis bias and 3D RMSE: planimetry and height behave differently
    in an aerial block, and a systematic bias means something quite different
    from a large spread.
    """
    banner(f"ACCURACY vs GROUND TRUTH{f' - {label}' if label else ''}")
    print(f"  {'':<22s}{'bias E':>8s}{'bias N':>9s}{'bias U':>9s}{'RMSE 3D':>11s}")
    print(f"  {'':<22s}{'(cm)':>8s}{'(cm)':>9s}{'(cm)':>9s}{'(cm)':>11s}")

    def axis_row(name, errors, note=""):
        e = np.asarray(errors, dtype=float).reshape(-1, 3)
        bias = e.mean(axis=0) * 100
        rmse3d = np.sqrt((e**2).sum(axis=1).mean()) * 100
        print(
            f"  {name:<22s}{bias[0]:+8.2f}{bias[1]:+9.2f}{bias[2]:+9.2f}"
            f"{rmse3d:11.2f}   {note}"
        )

    # If the bundle was given camera priors, this row is a FIT to those priors,
    # not an accuracy figure - say so, so it is never read as accuracy.
    axis_row(
        f"camera centres n={reconstruction.num_reg_images()}",
        [
            im.projection_center() - truth.camera_xyz[im.name]
            for im in reconstruction.images.values()
        ],
        "(fit - priors used)" if globals().get("USE_CAMERA_PRIORS") else "",
    )

    if truth.control_names is None:
        print("\n  no control/check split - load with split_control_check=True")
    else:
        for name, names, note in [
            ("control (GCP)", truth.control_names, "(fit - always flattering)"),
            ("check (CP)", truth.check_names, "<- TRUE ACCURACY"),
        ]:
            present = [n for n in names if n in marker_xyz]
            if present:
                axis_row(
                    f"{name} n={len(present)}",
                    [marker_xyz[n] - truth.marker_xyz[n] for n in present],
                    note,
                )

    # Rotation error reveals a residual tilt, which is invisible in the centre
    # residuals alone.
    angles = [
        np.degrees(
            np.arccos(
                np.clip(
                    (
                        np.trace(
                            im.cam_from_world().rotation.matrix()
                            @ true_cam_from_world(im.name).rotation.matrix().T
                        )
                        - 1
                    )
                    / 2,
                    -1,
                    1,
                )
            )
        )
        for im in reconstruction.images.values()
    ]
    print(
        f"\n  camera rotations       mean {np.mean(angles):.4f} deg   "
        f"max {np.max(angles):.4f} deg"
    )

    focal = next(iter(reconstruction.cameras.values())).params[0]
    print(
        f"  focal                  {focal:.3f}  vs reference "
        f"{enrich_data.FOCAL_PX:.3f}  ({focal - enrich_data.FOCAL_PX:+.3f})"
    )

In [12]:
banner("GCP-controlled bundle adjustment")

refine_intrinsics = True
MAX_SOLVE_RESTARTS = 3

# Constrain the camera centres to `camera_centers` as well as the GCPs. This is
# what Metashape's *_GT project does (camera_location_accuracy = 1 cm, all 39
# references enabled). Note the resulting camera-centre error is then a FIT to
# data the bundle was given, not an independent accuracy figure - the honest
# configuration is False.
USE_CAMERA_PRIORS = True
camera_prior_sigma_m = camera_position_sigma_m  # 0.01 to match Metashape's _GT

# COLMAP adds tie-point reprojections UNWEIGHTED, i.e. with an implicit
# sigma of 1 px. Only ratios between sigmas matter, so we express every other
# sigma in units of the tie-point sigma; that makes the whole problem a
# consistently whitened least-squares system without touching COLMAP's blocks.
image_cov = np.diag([(gcp_sigma_px / tie_points_sigma_px) ** 2] * 2)
world_cov = np.diag((gcp_sigma_m / tie_points_sigma_px) ** 2)
print(f"tie points      : sigma {tie_points_sigma_px} px  (reference unit)")
print(
    f"GCP marks       : sigma {gcp_sigma_px} px -> "
    f"{gcp_sigma_px / tie_points_sigma_px:.1f} in tie units"
)
print(
    f"GCP coordinates : sigma {gcp_sigma_m} m -> "
    f"{gcp_sigma_m / tie_points_sigma_px} in tie units"
)

config = pycolmap.BundleAdjustmentConfig()
for image_id in rec.reg_image_ids():
    config.add_image(image_id)
if not refine_intrinsics:
    for camera_id in rec.cameras:
        config.set_constant_cam_intrinsics(camera_id)

options = pycolmap.BundleAdjustmentOptions()
options.refine_focal_length = refine_intrinsics
options.refine_principal_point = False

# --- same tuning surface as the mapper cell ---------------------------------
# Points with fewer rays are IGNORED by the solver, not deleted: they keep
# pre-bundle coordinates and still reach the export, so pair this with the
# post-solve filter below. 0 = disabled; 3 drops the 2.5% two-ray points.
options.min_track_length = 0
# Robustification of the ~221k tie-point residuals. COLMAP uses SOFT_L1 @ 1.0
# for LOCAL BA and TRIVIAL for GLOBAL BA, so TRIVIAL matches its global default.
# SOFT_L1 down-weights the reprojection tail (p95 1.77, p99 2.27, max 3.98 px)
# without discarding redundancy - the soft alternative to hard filtering.
options.ceres.loss_function_type = pycolmap.LossFunctionType.TRIVIAL
options.ceres.loss_function_scale = 1.0
# COLMAP ships function_tolerance = 0 and parameter_tolerance = 0, which
# *disables* both relative stopping tests, leaving only the absolute
# gradient_tolerance = 1e-4. This block never reaches it, so every solve runs
# to the iteration cap and reports NO_CONVERGENCE even though the cost stopped
# improving. Re-enable the relative-cost test.
options.ceres.solver_options.max_num_iterations = 200
options.ceres.solver_options.function_tolerance = 1e-8

# IMPORTANT: create the adjuster ONCE, then add the GCP blocks to *its* problem
# and solve *it*. Building a second adjuster over the same config/rec yields a
# different ceres::Problem, silently orphaning every GCP block - the solve then
# runs on tie points alone and the control never enters the adjustment.
ba = pycolmap.create_default_ceres_bundle_adjuster(options, config, rec)
problem = ba.problem
n_tiepoint_blocks = problem.num_residual_blocks()

by_name = {im.name: im for im in rec.images.values()}
# A GCP becomes a parameter block we own; no Point3D/Track surgery needed.
marker_xyz = {name: np.array(surveyed_xyz[name], dtype=float) for name in marker_names}
# pyceres does NOT take ownership of cost functions - without this list the
# solver reads freed memory.
retained = []
robust = pyceres.CauchyLoss(2.8)

n_img_blocks = 0
for _, mark in marker_data.iterrows():
    image = by_name.get(mark.image_name)
    if image is None:
        continue
    camera = rec.cameras[image.camera_id]
    cost = cost_functions.ReprojErrorCost(
        camera.model, image_cov, np.array([mark.u, mark.v])
    )
    retained.append(cost)
    problem.add_residual_block(
        cost,
        robust,
        [
            marker_xyz[mark.marker_name],
            rec.frames[image.frame_id].rig_from_world.params,
            camera.params,
        ],
    )
    n_img_blocks += 1

for name in control_names:
    cost = pyceres.factors.NormalPrior(surveyed_xyz[name], world_cov)
    retained.append(cost)
    problem.add_residual_block(cost, None, [marker_xyz[name]])

# Camera position priors. AbsolutePosePositionPriorCost is the right cost here:
# its residual evaluates to R * (C_prior - C_est), so the sigma is metres on the
# camera CENTRE. A NormalPrior on rig_from_world.params would instead constrain
# the translation vector t, which is not the centre (C = -R^T t).
n_cam_blocks = 0
if USE_CAMERA_PRIORS:
    camera_cov = np.diag([(camera_prior_sigma_m / tie_points_sigma_px) ** 2] * 3)
    for image in rec.images.values():
        cost = cost_functions.AbsolutePosePositionPriorCost(
            camera_cov, camera_centers[image.name]
        )
        retained.append(cost)
        problem.add_residual_block(
            cost, None, [rec.frames[image.frame_id].rig_from_world.params]
        )
        n_cam_blocks += 1
    print(
        f"camera priors   : sigma {camera_prior_sigma_m} m -> "
        f"{camera_prior_sigma_m / tie_points_sigma_px} in tie units"
    )

expected = n_tiepoint_blocks + n_img_blocks + len(control_names) + n_cam_blocks
assert problem.num_residual_blocks() == expected, (
    f"{problem.num_residual_blocks()} residual blocks, expected {expected} - "
    "the GCP blocks are not in the problem being solved"
)
print(
    f"  added {n_img_blocks} GCP image residuals, "
    f"{len(control_names)} world priors, {n_cam_blocks} camera priors "
    f"({len(check_names)} check GCPs get rays but no prior)"
)
print(
    f"  problem: {problem.num_residual_blocks()} blocks "
    f"= {n_tiepoint_blocks} tie points + "
    f"{n_img_blocks + len(control_names) + n_cam_blocks} GCP/camera"
)

for attempt in range(1, MAX_SOLVE_RESTARTS + 1):
    summary = ba.solve()
    print(f"  solve {attempt}: {summary.termination_type}")
    if summary.termination_type == pycolmap.BundleAdjustmentTerminationType.CONVERGENCE:
        break

# --- post-solve filtering, same thresholds as the mapper cell ---------------
# filter_all_points3D also removes negative-depth points. min_tri_angle is in
# DEGREES (converted internally with DegToRad). Both are no-ops on this block,
# kept explicit so the thresholds are visible in one place.
filter_max_reproj_error = 4.0  # no-op here: max observed 3.98 px
filter_min_tri_angle = 1.5  # degrees; no-op here, p1 is 7.4 deg
min_track_length_filter = 0  # 0 = keep all; 3 removes the two-ray points

observation_manager = pycolmap.ObservationManager(rec)
n_filtered = observation_manager.filter_all_points3D(
    filter_max_reproj_error, filter_min_tri_angle
)
if min_track_length_filter > 0:
    n_filtered += observation_manager.filter_points3D_with_short_tracks(
        min_track_length_filter
    )
print(f"  filtered {n_filtered} observations -> {rec.num_points3D():,} points remain")

# Save the reconstruction to disk
sfm_gcp_path.mkdir(exist_ok=True, parents=True)
rec.write(sfm_gcp_path)
rec.write_text(sfm_gcp_path)
rec.export_PLY(sfm_gcp_path / "sparse_gcp.ply")


block_report(rec, marker_data, marker_xyz)
accuracy_report(rec, marker_xyz, truth, "after GCP bundle")



GCP-controlled bundle adjustment
tie points      : sigma 0.5 px  (reference unit)
GCP marks       : sigma 1.0 px -> 2.0 in tie units
GCP coordinates : sigma [0.02 0.02 0.03] m -> [0.04 0.04 0.06] in tie units
camera priors   : sigma 0.1 m -> 0.2 in tie units
  added 156 GCP image residuals, 18 world priors, 39 camera priors (8 check GCPs get rays but no prior)
  problem: 221904 blocks = 221691 tie points + 213 GCP/camera
  solve 1: BundleAdjustmentTerminationType.CONVERGENCE
  filtered 0 observations -> 43,754 points remain

BLOCK QUALITY
  images            39 registered / 39 in block | 1 camera(s) SIMPLE_PINHOLE
  intrinsics        [5865.136 3008.    2008.   ]
  points3D          43,754      observations 221,691
  track length      mean 5.07   median 4
                    2: 1,106 (2.5%)   3-5: 30,335 (69.3%)   6-9: 8,259 (18.9%)   10+: 4,054 (9.3%)
  observations/img  min 3,413   mean 5,684   max 7,027

  tie-point reprojection error (px)
    mean 0.607   median 0.411   RMS 0.817  

I20260812 16:27:28.643321 558585 bundle_adjustment_ceres.cc:1136] Bundle adjustment report
    Residuals : 443865
   Parameters : 131575
   Iterations : 2
         Time : 0.497401 [s]
 Initial cost : 0.444243 [px]
   Final cost : 0.444242 [px]
  Termination : CONVERGENCE



In [24]:
from custom_ba import (
    CameraPriorConstraints,
    FilterOptions,
    MarkerConstraints,
    SolverOptions,
    run_bundle_adjustment,
)

solver_options = SolverOptions()

marker_constraints = MarkerConstraints(
    observations=marker_data,
    xyz=marker_xyz,
    sigma_px=1.0,
    sigma_m=1.0,
    robust_loss_scale=2.8,
)

camera_prior_constraints = CameraPriorConstraints(
    xyz=camera_centers,
    sigma_m=camera_prior_sigma_m,
)

filter_options = FilterOptions(
    max_reproj_error=4.0,
    min_tri_angle_deg=1.5,
    min_track_length=min_track_length_filter,
)


ba_results = run_bundle_adjustment(
    reconstruction=rec,
    solver=solver_options,
    markers=marker_constraints,
    camera_priors=camera_prior_constraints,
    filters=filter_options,
    tie_points_sigma_px=tie_points_sigma_px,
    image_ids=None,  # None = all images
    verbose=True,
)

tie points      : sigma 0.5 px  (reference unit)
marker image obs: sigma 1.0 px -> 2.00 in tie units
marker world    : sigma 1.0 m -> 2.0 in tie units
camera priors   : sigma 0.1 m -> 0.2 in tie units
  problem: 221,912 blocks = 221,691 tie points + 221 added (156 marker image, 26 marker world, 39 camera)
  solve 1: BundleAdjustmentTerminationType.CONVERGENCE
  filtered 0 observations -> 43,754 points remain


In [26]:
rec = ba_results.reconstruction
block_report(rec, marker_data, marker_xyz)
accuracy_report(rec, marker_xyz, truth, "after GCP bundle")



BLOCK QUALITY
  images            39 registered / 39 in block | 1 camera(s) SIMPLE_PINHOLE
  intrinsics        [5865.234 3008.    2008.   ]
  points3D          43,754      observations 221,691
  track length      mean 5.07   median 4
                    2: 1,106 (2.5%)   3-5: 30,335 (69.3%)   6-9: 8,259 (18.9%)   10+: 4,054 (9.3%)
  observations/img  min 3,413   mean 5,684   max 7,027

  tie-point reprojection error (px)
    mean 0.607   median 0.411   RMS 0.817   p90 1.456   p99 2.269   max 3.98
    > 2 px: 1,130 points (2.6%)

  redundancy        443,382 measurements - 131,499 unknowns  ->  r = 311,883

  marker observations (156 in 39 images)
    reprojection  mean 0.24 px   median 0.22   max 0.55
    worst:  gcp.018 0.55 px,  gcp.007 0.54 px,  gcp.008 0.51 px,  gcp.023 0.46 px,  gcp.022 0.41 px

ACCURACY vs GROUND TRUTH - after GCP bundle
                          bias E   bias N   bias U    RMSE 3D
                            (cm)     (cm)     (cm)       (cm)
  camera centres n=3

In [ ]:
image = rec.images[1]
print(f"image {image.name} projection center: {image.projection_center()}")

In [ ]:
# Filter out points with too few observations or high reprojection error. This is a common post-processing step to clean up the reconstruction.
# rec = pycolmap.Reconstruction(sfm_gcp_path)
# drop = [pid for pid, p in rec.points3D.items() if p.track.length() < 3 or p.error > 2.0]
# for pid in drop:
#     rec.delete_point3D(pid)

In [ ]:
def save_point_cloud_as_ply(reconstruction, output_path) -> bool:
    """Save the 3D points of the reconstruction as a PLY file."""
    import open3d as o3d

    points = list(reconstruction.points3D.values())
    pc = o3d.t.geometry.PointCloud()
    pc.point.positions = o3d.core.Tensor(np.array([p.xyz for p in points], np.float32))
    pc.point.colors = o3d.core.Tensor(np.array([p.color for p in points], np.uint8))
    pc.point.track_length = o3d.core.Tensor(
        np.array([[p.track.length()] for p in points], np.int32)
    )
    pc.point.reproj_error = o3d.core.Tensor(
        np.array([[p.error] for p in points], np.float32)
    )
    return o3d.t.io.write_point_cloud(output_path, pc)


save_point_cloud_as_ply(rec, sfm_gcp_path / "sparse_gcp_o3d.ply")

True